# 01 — Dataset inspection
Results produced by `scripts/inspect_dataset.py`; explanation in `docs/01_dataset_inspection.md`.
This notebook only **loads and displays** saved, real outputs — the logic lives in `src/predictops/` and `scripts/`.

In [1]:
import json, pandas as pd
from IPython.display import Image, display, Markdown
from predictops.config import project_path
pd.set_option('display.max_columns', 40)
R = project_path('reports'); F = R / 'figures'
def show(p, w=950):
    display(Image(filename=str(p), width=w))

In [2]:
from predictops.data.ingestion import load_intermediate
df = load_intermediate()
display(df.head()); df.dtypes

,raw_index,timestamp,TP2,TP3,H1,DV_pressure,Reservoirs,Oil_temperature,Motor_current,COMP,DV_eletric,Towers,MPG,LPS,Pressure_switch,Oil_level,Caudal_impulses,dt_seconds,segment_id
0,0,2020-02-01 00:00:00,-0.012,9.358,9.340,-0.024,9.358,53.599998,0.0400,1,0,1,1,0,1,1,1,0.0,0
1,10,2020-02-01 00:00:10,-0.014,9.348,9.332,-0.022,9.348,53.674999,0.0400,1,0,1,1,0,1,1,1,10.0,0
2,20,2020-02-01 00:00:19,-0.012,9.338,9.322,-0.022,9.338,53.599998,0.0425,1,0,1,1,0,1,1,1,9.0,0
3,30,2020-02-01 00:00:29,-0.012,9.328,9.312,-0.022,9.328,53.424999,0.0400,1,0,1,1,0,1,1,1,10.0,0
4,40,2020-02-01 00:00:39,-0.012,9.318,9.302,-0.022,9.318,53.474998,0.0400,1,0,1,1,0,1,1,1,10.0,0


raw_index                   int64
timestamp          datetime64[us]
TP2                       float32
TP3                       float32
H1                        float32
DV_pressure               float32
Reservoirs                float32
Oil_temperature           float32
Motor_current             float32
COMP                         int8
DV_eletric                   int8
Towers                       int8
MPG                          int8
LPS                          int8
Pressure_switch              int8
Oil_level                    int8
Caudal_impulses              int8
dt_seconds                float32
segment_id                  int32
dtype: object

In [3]:
display(Markdown(project_path('docs/data_dictionary.md').read_text()))

# Data Dictionary — MetroPT-3

_Meanings from the official `Data Description_Metro.pdf`; ranges computed from the data by `scripts/inspect_dataset.py`._

| Column | Type | Unit | Stored dtype | Observed range | Meaning |
|---|---|---|---|---|---|
| `Unnamed: 0` → `raw_index` | index | – | int64 | 0 … 15,169,470 (step 10) | Row number in the original 1 Hz log; used only for traceability, never as a feature |
| `timestamp` | datetime | – | datetime64 | 2020-02-01 00:00:00 … 2020-09-01 03:59:50 | Time of the reading (~10 s steps) |
| `TP2` | analog | bar | float32 | -0.032 … 10.676 | Pressure on the compressor. |
| `TP3` | analog | bar | float32 | 0.730 … 10.302 | Pressure generated at the pneumatic panel. |
| `H1` | analog | bar | float32 | -0.036 … 10.288 | Pressure generated by the pressure drop when the cyclonic separator filter discharges. |
| `DV_pressure` | analog | bar | float32 | -0.032 … 9.844 | Pressure drop when the towers discharge the air dryers; ~0 means the compressor is operating under load. |
| `Reservoirs` | analog | bar | float32 | 0.712 … 10.300 | Downstream reservoir pressure; should be close to TP3. |
| `Oil_temperature` | analog | °C | float32 | 15.400 … 89.050 | Oil temperature on the compressor. |
| `Motor_current` | analog | A | float32 | 0.020 … 9.295 | Current of one motor phase: ~0 A off, ~4 A offloaded, ~7 A under load, ~9 A at start. |
| `COMP` | digital | 0/1 | int8 | 0.000 … 1.000 | Air intake valve signal; active when there is no air intake (compressor off or offloaded). |
| `DV_eletric` | digital | 0/1 | int8 | 0.000 … 1.000 | Compressor outlet valve signal; active when under load. (Spelling as in the dataset.) |
| `Towers` | digital | 0/1 | int8 | 0.000 … 1.000 | Which air-drying tower is active: 0 = tower one, 1 = tower two. |
| `MPG` | digital | 0/1 | int8 | 0.000 … 1.000 | Starts the compressor under load when APU pressure < 8.2 bar; drives COMP. |
| `LPS` | digital | 0/1 | int8 | 0.000 … 1.000 | Low-pressure switch: activates when pressure drops below 7 bar. |
| `Pressure_switch` | digital | 0/1 | int8 | 0.000 … 1.000 | Detects discharge in the air-drying towers. |
| `Oil_level` | digital | 0/1 | int8 | 0.000 … 1.000 | Active when oil is below expected level. |
| `Caudal_impulses` | digital | 0/1 | int8 | 0.000 … 1.000 | Pulse output counting air flow from the APU to the reservoirs. |

## Derived columns (added by PredictOps, not in raw data)

| Column | Meaning |
|---|---|
| `dt_seconds` | Seconds since previous reading |
| `segment_id` | Continuous recording segment; increments after a gap > 600 s |
